# فاین‌تیون فارسی Chatterbox — Colab

اجرای همان pipeline مخزن روی Colab یا هر سرور GPU اجاره‌ای.

### دو اصلی که این نوت‌بوک روی آن‌ها بنا شده

**۱. هر چیز باارزشی در Google Drive می‌ماند.** جلسه‌ی Colab هر لحظه ممکن است قطع شود و
دیسکش پاک می‌گردد. وزن‌های مدل، کش داده و مهم‌تر از همه checkpointها روی Drive نوشته
می‌شوند، پس یک قطعی فقط چند دقیقه هزینه دارد نه کل اجرا.

**۲. preprocessing را محلی انجام دهید.** خروجی‌اش حدود **۵.۵ کیلوبایت به‌ازای هر کلیپ**
است (~۳۷۵ مگابایت برای کل ۶۸ هزار کلیپ) در برابر ده‌ها گیگابایت صدای خام. آپلود کش
به‌جای صدا، هم آپلود را کوچک می‌کند و هم ساعت GPU اجاره‌ای را صرف آموزش می‌کند نه decode.


## ۱. بررسی GPU

دقت عددی خودکار انتخاب می‌شود: `bf16` روی Ampere به بعد (A100 / L4 / 4090)
و `fp16` روی Turing. T4 معماری Turing است و bf16 سخت‌افزاری ندارد.


In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f'{torch.cuda.get_device_name(0)}  capability {major}.{minor}  {total:.1f} GB')
    print('native bf16:', major >= 8)
else:
    print('No GPU. Runtime > Change runtime type > GPU')


## ۲. اتصال Google Drive

همه‌چیز زیر یک پوشه جمع می‌شود تا اجراهای بعدی همان‌ها را پیدا کنند.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/chatterbox-fa')
for sub in ['pretrained_models', 'cache', 'output']:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

print('Drive workspace:', DRIVE)
!df -h /content/drive | tail -1


## ۳. آوردن پروژه


In [ ]:
REPO = 'https://github.com/TheServat/chatterbox-finetuning-persian.git'
PROJECT = '/content/chatterbox-finetuning-persian'

import os, pathlib
os.chdir('/content')
if not pathlib.Path(PROJECT).exists():
    !git clone -q $REPO

# os.chdir rather than %cd: a line magic inside an if-block is awkward, and
# every ! command below inherits this working directory anyway.
os.chdir(PROJECT)
print(os.getcwd())
!git log --oneline -1


## ۴. نصب وابستگی‌ها

پکیج `chatterbox-tts` عمداً نصب نمی‌شود: سورس آن در `src/chatterbox_` هست و
نصب همزمان دو نسخه را روی مسیر می‌گذارد و معلوم نمی‌شود کدام اجرا می‌شود.


In [ ]:
!pip install -q -r requirements.txt

import importlib
for module in ['torch', 'transformers', 'peft', 'soundfile', 'num2words', 'pyarrow']:
    print(f'{module:14s}', importlib.import_module(module).__version__)


## ۵. ورود به HuggingFace

`login()` توکن را در `~/.cache/huggingface` می‌گذارد و همه‌ی ابزارها بدون نیاز به
متغیر محیطی از آن استفاده می‌کنند. توکن داخل فایل نوت‌بوک ذخیره نمی‌شود.

اگر توکن را در پنل 🔑 **Secrets** با نام `HF_TOKEN` گذاشته باشید، سلول بدون
پرسش وارد می‌شود.


In [ ]:
from huggingface_hub import login, whoami

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass

# add_to_git_credential=False keeps the token out of any repo you clone here.
login(token=token, add_to_git_credential=False)
print('logged in as:', whoami()['name'])


In [ ]:
# The project's tools read HF_TOKEN from the environment, so hand them the
# token the login just stored.
import os
from huggingface_hub import HfFolder
os.environ['HF_TOKEN'] = HfFolder.get_token()
print('HF_TOKEN exported for the CLI tools:', bool(os.environ['HF_TOKEN']))


## ۶. وزن‌های مدل — یک‌بار دانلود، ماندگار روی Drive

`pretrained_models/` به Drive لینک می‌شود، پس این ۳.۲ گیگ فقط در اولین اجرا
دانلود می‌شود و جلسه‌های بعدی مستقیم از Drive می‌خوانند.

نسخه‌ها از `versions.lock.json` می‌آیند، پس هر اجرا دقیقاً همان artifactها را می‌گیرد.
این مرحله توکنایزر `[fa]` را هم می‌سازد.


In [ ]:
import pathlib, shutil

local = pathlib.Path('pretrained_models')
if local.is_symlink():
    pass
else:
    if local.exists():
        shutil.rmtree(local)
    local.symlink_to(DRIVE / 'pretrained_models', target_is_directory=True)
print('pretrained_models ->', local.resolve())


In [ ]:
!python tools/fetch_models.py
!ls -la pretrained_models/


## ۷. داده

**مسیر الف (توصیه‌شده)** — کش preprocess را محلی بسازید و آپلود کنید:

```bash
python tools/build_dataset.py --dedupe
python -m src.preprocess_ljspeech
tar -czf preprocess_fa.tar.gz -C MyTTSDataset preprocess
```

فایل را در `MyDrive/chatterbox-fa/cache/` بگذارید.

> کش **روی دیسک محلی باز می‌شود، نه روی Drive**: ۶۸ هزار فایل کوچک را از Drive
> خواندن، DataLoader را خفه می‌کند. آرشیو روی Drive می‌ماند، محتوایش محلی.


In [ ]:
import pathlib, subprocess

ARCHIVE = DRIVE / 'cache' / 'preprocess_fa.tar.gz'
target = pathlib.Path('MyTTSDataset')

if ARCHIVE.exists():
    target.mkdir(exist_ok=True)
    print(f'extracting {ARCHIVE.stat().st_size/2**20:.0f} MB to local disk...')
    subprocess.run(['tar', '-xzf', str(ARCHIVE), '-C', str(target)], check=True)
    n = len(list((target / 'preprocess').glob('*.pt')))
    print(f'{n:,} preprocessed clips ready')
else:
    print(f'{ARCHIVE} not found - upload it, or use path B below.')


**مسیر ب** — ساخت روی Colab. فقط وقتی سیستم محلی GPU ندارد:
YodaLingua حدود ۱ گیگ دانلود و ۱۲ گیگ دیسک برای wav می‌خواهد، و preprocessing هم زمان می‌برد.

در پایان حتماً کش را روی Drive ذخیره کنید تا جلسه‌ی بعد دوباره لازم نباشد.


In [ ]:
# Skip entirely if path A worked.
!python tools/fetch_datasets.py yoda
!python tools/build_dataset.py --sources yoda --dedupe
!python -m src.preprocess_ljspeech

# Then back it up, so a disconnect does not cost this work again.
!tar -czf /content/preprocess_fa.tar.gz -C MyTTSDataset preprocess
!cp /content/preprocess_fa.tar.gz "$DRIVE/cache/"
print('cache saved to Drive')


## ۸. خروجی روی Drive

`chatterbox_output/` هم به Drive لینک می‌شود، پس checkpointها همان لحظه‌ی ذخیره
روی Drive می‌نشینند. اگر جلسه قطع شود، با `--resume` از آخرین checkpoint ادامه می‌دهید.


In [ ]:
import pathlib, shutil

out = pathlib.Path('chatterbox_output')
if not out.is_symlink():
    if out.exists():
        shutil.rmtree(out)
    out.symlink_to(DRIVE / 'output', target_is_directory=True)
print('chatterbox_output ->', out.resolve())
!ls -la chatterbox_output/


## ۹. آموزش

اول یک smoke-test کوتاه، تا پیش از شروع اجرای چندساعته مطمئن شویم مسیر سالم است
و ببینیم چقدر VRAM مصرف می‌شود.


In [ ]:
!python train.py --max-steps 5 --batch-size 4 --grad-accum 1 --workers 2 --no-preprocess


اگر پیک VRAM خیلی کمتر از ظرفیت کارت بود `--batch-size` را بالا ببرید:
روی T4 (۱۶ گیگ) معمولاً ۱۶ و روی A100 یا 4090 حدود ۳۲ جا می‌شود.
`--grad-accum` را طوری بگذارید که batch مؤثر (`batch_size × grad_accum`) حدود ۳۲ بماند.

`--save-steps 200` یعنی حداکثر ۲۰۰ گام کار در یک قطعی از دست می‌رود.
`--sample` در هر checkpoint یک نمونه‌ی صوتی فارسی می‌سازد تا پیشرفت را **بشنوید**،
نه اینکه فقط از منحنی loss حدس بزنید.


In [ ]:
# One line on purpose: backslash continuation in a ! command is not
# reliably supported by IPython.
!python train.py --batch-size 16 --grad-accum 2 --workers 2 --save-steps 200 --sample --resume --no-preprocess


> اگر جلسه قطع شد: از سلول ۲ تا اینجا را دوباره اجرا کنید. همان دستور با `--resume`
> آخرین checkpoint را در Drive پیدا می‌کند و از همان‌جا ادامه می‌دهد.


### پایش با TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir chatterbox_output/runs


نمونه‌های صوتی تولیدشده حین آموزش:


In [ ]:
import pathlib
from IPython.display import Audio, display

samples = sorted(pathlib.Path('chatterbox_output/inference_samples').glob('*.wav'))
for path in samples[-3:]:
    print(path.name)
    display(Audio(str(path)))


## ۱۰. تولید صدا

`--long` متن را روی مرزهای جمله‌ی فارسی می‌شکند و تکه‌ها را به هم می‌چسباند.
بدون آن هر فراخوانی حداکثر حدود ۴۰ ثانیه صدا می‌دهد (سقف ۱۰۰۰ توکن گفتاری در ۲۵ هرتز).


In [ ]:
TEXT = 'سلام، این یک آزمایش برای مدل گفتار فارسی است. امیدوارم صدای طبیعی و روانی داشته باشد.'

!python infer_fa.py --text "$TEXT" --out sample.wav

from IPython.display import Audio, display
display(Audio('sample.wav'))


In [ ]:
long_text = '''هوش مصنوعی در سال‌های اخیر پیشرفت چشمگیری داشته است.
مدل‌های زبانی بزرگ توانسته‌اند در ترجمه، خلاصه‌سازی و تولید متن به نتایج قابل توجهی برسند؛
با این حال چالش‌هایی مانند سوگیری و نیاز به داده‌های باکیفیت همچنان باقی است.'''

open('long.txt', 'w', encoding='utf-8').write(long_text)
!python infer_fa.py --text-file long.txt --long --out long.wav

from IPython.display import Audio, display
display(Audio('long.wav'))


## ۱۱. نتیجه‌ی نهایی

آداپتور از قبل روی Drive است (چون `chatterbox_output` لینک شده). این سلول فقط
تأیید می‌کند و اندازه را نشان می‌دهد.

آداپتور به همان مدل پایه‌ای که رویش آموزش دیده گره خورده و روی پایه‌ی دیگری بارگذاری نمی‌شود.


In [ ]:
!du -sh chatterbox_output/persian_adapter 2>/dev/null || echo 'not trained yet'
!ls -la "$DRIVE/output/"


---

### Colab یا اجاره‌ی GPU؟

| | Colab Pro | Colab Pro+ | RunPod / Vast (4090) |
|---|---|---|---|
| GPU | T4 ۱۶ گیگ | گاهی A100 | 4090 ۲۴ گیگ |
| bf16 سخت‌افزاری | ندارد (Turing) | دارد | دارد |
| محدودیت زمان | ~۱۲ ساعت، با قطعی | ~۲۴ ساعت | ندارد |
| دیسک ماندگار | ندارد (Drive لازم است) | ندارد | volume دارد |
| هزینه | ~۱۰$ ماهانه | ~۵۰$ ماهانه | ~۰.۴$ ساعتی |

برای یک اجرای کامل، اجاره‌ی 4090 معمولاً هم ارزان‌تر درمی‌آید و هم قطع نمی‌شود.
Colab برای آزمایش سریع و تنظیم hyperparameter عالی است — و با `--resume` و
خروجی روی Drive، یک اجرای طولانی هم شدنی است، فقط با چند بار وصل شدن دوباره.
